# 74. 원판 검출에서 200에폭까지 기다린 결과

조기 종료를 끈 한 번의 선형 학습에서 106·150·200에폭을 미리 지정했습니다. 기존 0.0005 코사인의 best.pt와 같은 검증 선수 6명을 비교합니다. 아래 셀은 Mac에 저장한 결과를 읽습니다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT

folder = ROOT / "outputs/plate_lr_200_checkpoints/20261126"
review = folder / "review"
summary = pd.read_csv(review / "summary.csv").set_index("checkpoint")
athletes = pd.read_csv(review / "athletes.csv")
gaps = pd.read_csv(review / "gaps.csv")
frames = pd.read_csv(review / "frames.csv")
display(summary.round(2))

## 에폭이 늘면서 검증 손실과 mAP가 어떻게 달라졌나

선은 원시 에폭 값입니다. 세로선은 가중치 저장 시점입니다.

In [ ]:
run = folder / "runs/lr0001_linear_200"
history = pd.read_csv(run / "results.csv").set_index("epoch")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
for ax, column in zip(axes, ["val/box_loss", "metrics/mAP50-95(B)"]):
    ax.plot(history[column], color="#29547a", linewidth=1.4)
    for epoch in [106, 150, 200]:
        ax.axvline(epoch, linestyle=":", color="gray")
        ax.scatter(epoch, history.at[epoch, column], color="#ad481e")
    ax.set(xlabel="Epoch", title=column)
fig.savefig(review / "checkpoint_curves.png", dpi=135)
plt.show()

## 선수별 허브 상승량과 좌표 누락

무릎 통과→접촉 상승량에서 수동값을 뺀 오차입니다. 양수는 과대, 음수는 과소 측정입니다. 누락 구간의 first와 last는 양 끝을 포함합니다.

In [ ]:
errors = athletes.pivot(index="video_id", columns="checkpoint", values="rise_error_px")[summary.index]
manual = athletes.drop_duplicates("video_id").set_index("video_id").manual_rise_px
display(errors.assign(manual_px=manual).round(2))
display(gaps)
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
errors.plot.bar(ax=ax, color=["#334155", "#6484ac", "#bc7241", "#357861"], rot=0)
ax.axhline(0, color="black", linewidth=0.7)
ax.set(ylabel="Predicted minus manual rise (px)", xlabel="", title="Knee pass to contact")
ax.legend(fontsize=8)
fig.savefig(review / "rise_errors.png", dpi=135)
plt.show()

## b02 접촉 36번의 원판 테두리와 모델 박스

연두색 점선은 수동 박스, 주황색은 시간 연결 후 선택된 박스, 흰 십자는 수동 허브입니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import ANNOTATIONS, VIDEOS
from wlcv.data.frames import read_rgb
video, index = "b02_sn_h_02", 36
rows = frames[(frames.video_id == video) & (frames.frame_index == index)].set_index("checkpoint")
hubs = pd.read_csv(ANNOTATIONS / "plate_blur_review/validation.csv")
hub = hubs[(hubs.video_id == video) & (hubs.frame_index == index)].iloc[0]
image = read_rgb(VIDEOS / f"{video}.mp4", index)
fig, axes = plt.subplots(1, 4, figsize=(11, 3.2), layout="constrained")
for ax, name in zip(axes, summary.index):
    r = rows.loc[name]
    ax.imshow(image)
    w, h = r.gt_x2-r.gt_x1, r.gt_y2-r.gt_y1
    ax.add_patch(Rectangle((r.gt_x1,r.gt_y1),w,h,fill=False,edgecolor="lime",linestyle="--",lw=2))
    if pd.notna(r.x1):
        ax.add_patch(Rectangle((r.x1,r.y1),r.x2-r.x1,r.y2-r.y1,fill=False,edgecolor="darkorange",lw=2))
    ax.plot(hub.manual_x,hub.manual_y,"w+",ms=9,mew=2)
    ax.set(xlim=(r.gt_x1-.3*w,r.gt_x2+.3*w),ylim=(r.gt_y2+.3*h,r.gt_y1-.3*h),title=f"{name}\nIoU {r.iou:.2f}")
    ax.axis("off")
fig.savefig(review / "b02_contact_local.png", dpi=135)
plt.show()